In [ ]:
import numpy as np 
import pandas as pd 

In [ ]:
#Loading data from CVS file

In [ ]:
data="Leads.csv"
df1=pd.read_csv(data)
df1.head()

Exploratory data analysis [EDA]

In [ ]:

df1.columns = df1.columns.str.lower().str.replace(' ', '_')

string_columns = list(df1.dtypes[df1.dtypes == 'object'].index)

for col in string_columns:
    df1[col] = df1[col].str.lower().str.replace(' ', '_')
df1.head()

In [ ]:
df1.dtypes

In [ ]:
df1.nunique()

In [ ]:
for cl in df1.columns:
    print(cl, df1[cl].nunique())
    print (df1[cl].unique())
    if df1[cl].nunique()==1:
        del(df1[cl])

    print()
    print()

In [ ]:
# removing these two columns 

In [ ]:
#del (df1['prospect_id'])
#del (df1['lead_number'])

In [ ]:
df1.head()

In [ ]:
print(df1.isnull().sum())
print(len(df1))

In [ ]:
for cl in df1.columns:
    print (cl)
    print(df1[cl].unique())
    print()
    print()
    

replacing all missing values with 0

In [ ]:
df_old=df1
for cl in df1.columns:
    if df1[cl].dtype==object:
        df1[cl]=df1[cl].fillna("other")
    else :
        df1[cl]=df1[cl].fillna(0) 
print(df1.isnull().sum())

look at the distribution of non object series 

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
numerical=[]
categorical=[]
for cl in df1.columns:
    if df1[cl].dtype!=object:
        print (cl)
        if cl!='converted':
            numerical.append(cl)
        sns.histplot(df1[cl][df1[cl]>0], bins=50)
    else :
        categorical.append(cl)
        



In [ ]:
sns.histplot(df1['totalvisits'][df1.totalvisits>0], bins=50)
#print(categorical)

In [ ]:
sns.histplot(df1['total_time_spent_on_website'][df.total_time_spent_on_website>0], bins=50)


In [ ]:
sns.histplot(df1['page_views_per_visit'], bins=50)

In [ ]:
sns.histplot(df1.asymmetrique_activity_score[df1.asymmetrique_activity_score>0], bins=50)

In [ ]:
sns.histplot(df1.asymmetrique_profile_score [df1.asymmetrique_profile_score>0], bins=50)

In [ ]:
df1.lead_profile.unique()

In [ ]:
from sklearn.metrics import mutual_info_score
def mutual_info_churn_score(series):
   return mutual_info_score(series,df.converted)
#mutual_info_score(df.lead_profile,df.converted)
mi = df1[numerical+categorical].apply(mutual_info_churn_score)
mi.sort_values(ascending=False)
mi


In [ ]:
print(mi.keys())


In [ ]:
selection=[ 'tags','totalvisits','total_time_spent_on_website','last_notable_activity', 'city', 'converted']

In [ ]:
import numpy as np

In [ ]:
df=df1[selection].loc[:]
#df['total_time_spent_on_website']=np.log1p( df['total_time_spent_on_website'])
df.head()
df.isnull().sum()

In [ ]:
mi = df[:].apply(mutual_info_churn_score)
mi.sort_values(ascending=True)
mi

In [ ]:
df.nunique()

In [ ]:
df.dtypes

In [ ]:
#df.total_time_spent_on_website[:]=pd.to_numeric(df.total_time_spent_on_website, errors='coerce')

In [ ]:
#df.converted=pd.to_numeric(df.converted, errors='coerce')

Spliting the data into train/validation/test

In [ ]:
from sklearn.model_selection import  train_test_split


train_full, df_test= train_test_split(df,test_size=0.2,shuffle=True)
df_train, df_validation= train_test_split(train_full,test_size=0.25,shuffle=True)
df_train_full=train_full.reset_index(drop=True)
df_train_full.reset_index(drop=True)
df_train.reset_index(drop=True)
df_validation.reset_index(drop=True)
df_test.reset_index(drop=True)

In [ ]:
print(df_train11.loc[0].to_json())

setting up y for different splits 

In [ ]:
y_train= df_train.converted.values
y_test=df_test.converted.values
y_val=df_validation.converted.values
y_train_full=df_train_full.converted.values
del df_train['converted']
del df_test['converted']
del df_validation['converted']
del  df_train_full['converted']

One hot encoding 

In [ ]:
from sklearn.feature_extraction import DictVectorizer

In [ ]:
train_dicts =df_train.to_dict(orient='records')
dv=DictVectorizer(sparse=False)
dv.fit(train_dicts) #show him how the data look like
x_train=dv.transform(train_dicts)

In [ ]:
x_train[0]

 Linear Regression LR 

In [ ]:
from sklearn.linear_model import LogisticRegression
model =LogisticRegression(max_iter=10000)
model.fit(x_train,y_train)
pred_train=model.predict(x_train)
print((pred_train==y_train).mean())
#print(model.score(pred_train,y_train))


In [ ]:
validation_dicts =df_validation.to_dict(orient='records')
x_validation=dv.transform(validation_dicts)
pred_validation=model.predict(x_validation)

print((pred_validation==y_val).mean())
print 


In [ ]:
print (pred_train, y_train)

In [ ]:
from sklearn.metrics import roc_auc_score
auc = roc_auc_score(y_val, pred_validation)
print(auc)

retrain the model with full_train 

In [ ]:
x_train_full=dv.transform(df_train_full.to_dict(orient='records'))
model.fit(x_train_full,y_train_full)
pred_train_full=model.predict(x_train_full)
print((pred_train_full==y_train_full).mean())

confusion matrix 


In [ ]:
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_val, pred_validation>=0.5)
print("Confusion Matrix:\n", cm)

precision recall

In [ ]:
from sklearn.metrics import precision_score, recall_score
precision = precision_score(y_val, pred_validation>0.5, average='binary')
recall = recall_score(y_val, pred_validation>00.5, average='binary')
print("Precision:", precision)
print("Recall:", recall)
print("F1 score:",2*(precision*recall/(precision+recall)))

ROC curves

In [ ]:
from sklearn.metrics import roc_curve
fpr, tpr, thresholds = roc_curve(y_val, pred_validation)
plt.figure(figsize=(5, 5))
plt.plot(fpr, tpr, label='Model')
plt.plot([0, 1], [0, 1], label='Random', linestyle='--')
plt.xlabel('FPR') 
plt.ylabel('TPR')
plt.legend()

cross validation 

In [ ]:
def train(df_train1, y_train, C=1.0):
   df_train=df_train1[selection1]
   dicts = df_train.to_dict(orient='records')
   dv1=DictVectorizer(sparse=False)
   dv1.fit(dicts) #show him how the data look like
   x_train=dv1.transform(dicts)

   model1 = LogisticRegression(C=C, max_iter=100000)
   model1.fit(x_train, y_train)
  
   return dv, model

def predict(df, dv, model):
   dicts = df[selection1].to_dict(orient='records')


   X = dv.transform(dicts)
   y_pred = model.predict_proba(X)[:, 1]


   return y_pred

In [ ]:
from sklearn.model_selection import KFold
from sklearn.metrics import roc_auc_score
n_splits = 5
selection1=[ 'tags','totalvisits','last_notable_activity', 'city']
for C in [0.001, 0.01, 0.1, 0.5, 1, 5, 10]:
   kfold = KFold(n_splits=n_splits, shuffle=True, random_state=1)
   scores = []
   for train_idx, val_idx in kfold.split(df_train_full):
       df_train11 = train_full.iloc[train_idx]
       df_val11 = train_full.iloc[val_idx]
       y_train11 = df_train11.converted.values
       y_val11 = df_val11.converted.values
       dv11, model11 = train(df_train11, y_train11)
       y_pred11 = predict(df_val11, dv11, model11)
       auc11 = roc_auc_score(y_val11, y_pred11)
       scores.append(auc11)
   print('C=%s %.3f +- %.3f' % (C, np.mean(scores), np.std(scores)))

test partition 

In [ ]:
x_test=dv.transform(df_validation.to_dict(orient='records'))
pred_test=model.predict_proba(x_test)[:, 1]
print("Accuracy test score ",(y_test==(pred_test>=0.5)).mean())
auc_test = roc_auc_score(y_test, pred_test)
print('AUC test score ',auc_test)
from sklearn.metrics import accuracy_score
accuracy_score( y_test,(pred_test >= 0.5))

Saving the model

In [ ]:
import pickle

In [ ]:
output_file=f'model_1.bin'
f_out=open(output_file,'wb')
pickle.dump((dv,model),f_out)
f_out.close()